# Finviz live HTML — Jupyter isolation

Same Elite session + parsers as the jobs that 403 on Aliyun ECS / sometimes on GitHub-hosted.

Auth (first match, same as src/finviz_session.py): FINVIZ_AUTH / AUTH_TOKEN_FINVIZ, else FINVIZ_EMAIL + FINVIZ_PASSWORD.
Does not write 01_daily/.


In [ ]:
import os, sys
from pathlib import Path
for cand in [Path.cwd(), Path.cwd().parent, Path('/home/gha/fullscan'), Path.home()/'fullscan']:
    if (cand / 'src' / 'finviz_session.py').exists():
        os.chdir(cand)
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        print('ROOT', cand)
        break
else:
    print('WARN: clone/cd into fullscan')
for env_path in [Path.home()/'.fullscan.env', Path('/home/gha/.fullscan.env'), Path('scripts/fullscan.env')]:
    if env_path.exists():
        print('loading', env_path)
        for line in env_path.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, _, v = line.partition('=')
            k, v = k.strip(), v.strip().strip('"').strip("'")
            if k and k not in os.environ:
                os.environ[k] = v
os.environ.setdefault('FINVIZ_GAP_SEC', '5')
print('gap', os.environ.get('FINVIZ_GAP_SEC'))
print('auth cookie set', bool((os.environ.get('FINVIZ_AUTH') or os.environ.get('AUTH_TOKEN_FINVIZ') or '').strip()))
print('email set', bool((os.environ.get('FINVIZ_EMAIL') or '').strip()))
print('password set', bool((os.environ.get('FINVIZ_PASSWORD') or '').strip()))


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
from src import finviz_session
from src.map_heat import (
    fetch_groups, fetch_futures, fetch_econ, fetch_earnings,
    fetch_stock_news, fetch_major_news_tickers, _tape_from_futures,
)
from src.finviz_digest import _scrape_index_digest, INDEX_TICKERS
from src.quote_colors import fetch_ticker
ET = ZoneInfo('America/New_York')
DATE = datetime.now(ET).date().isoformat()
print('ET date', DATE)
sess = finviz_session.session()
print('authed', finviz_session.authed(sess), 'flag', sess.headers.get('X-Fullscan-Finviz'))


## 1. Overlay paths (`map_heat --overlay`)


In [ ]:
futures = fetch_futures(sess)
tape = _tape_from_futures(futures)
print('futures tiles', len(futures), 'tape kept', len(tape))
display(pd.DataFrame(tape))
econ = fetch_econ(sess, DATE)
print('econ', DATE, len(econ))
display(pd.DataFrame(econ) if econ else pd.DataFrame(columns=['event']))
earns = fetch_earnings(sess, DATE)
print('earnings', DATE, len(earns))
display(pd.DataFrame(earns)[:15] if earns else pd.DataFrame(columns=['ticker']))
news = fetch_stock_news(sess)
print('ticker news', len(news))
display(pd.DataFrame(news)[:15] if news else pd.DataFrame())
maj = fetch_major_news_tickers(sess)
print('major-news', len(maj), maj[:20])


## 2. Groups (`map_heat --force` on ECS)


In [ ]:
ind = fetch_groups(sess, 'industry')
sec = fetch_groups(sess, 'sector')
print('industries', len(ind), 'sectors', len(sec))
display(pd.DataFrame.from_dict(sec, orient='index').rename_axis('sector').reset_index())
display(pd.DataFrame.from_dict(ind, orient='index').rename_axis('industry').reset_index().head(20))


## 3. Digest live quotes (`finviz_digest`)


In [ ]:
rows = []
for t in INDEX_TICKERS:
    row = _scrape_index_digest(t, sess, export=None)
    rows.append(row or {'ticker': t, 'error': 'none'})
    print(t, (row or {}).get('source'), (row or {}).get('digest') or (row or {}).get('error'))
display(pd.DataFrame(rows))


## 4. Quote colors (SPY + AAPL smoke)


In [ ]:
color_rows = []
for t in ['SPY', 'AAPL']:
    try:
        data = fetch_ticker(t, sess)
        data = {k: v for k, v in data.items() if k != 'fields'}
        color_rows.append(data)
        print(t, 'green', data['n_green'], 'red', data['n_red'], 'delta', data['green_minus_red'])
    except Exception as e:
        print(t, 'FAIL', e)
        color_rows.append({'ticker': t, 'error': str(e)})
display(pd.DataFrame(color_rows))
